# W06B 실습 B · rank-k 재구성, 추천의 오해와 웹 앱

딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2

2026년 10월 8일. 실습 A를 실행하지 않았어도 독립 실행됩니다. 목표: 큰 패턴의 합과 오차를 계산하고, 입력 재구성과 숨긴 평점의 예측을 구별하며, 마지막 앱의 설명 문장을 직접 수정합니다.


## 실행과 코드 읽기

CPU 런타임에서 첫 셀부터 순서대로 실행합니다. GPU·업로드·외부 데이터는 필요 없습니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요. 설치 후 재시작 안내가 뜨면 재시작 후 첫 셀부터 다시 실행합니다.

빈칸 변수와 자동 실험은 분리되어 있습니다. 전체 실행 성공은 자신의 빈칸·서술 답이 정답이라는 뜻이 아닙니다. `None`과 빈 문자열을 자신의 계산·설명으로 바꾸세요. 수치 배열의 미세한 오차는 정상이며 `np.allclose`로 비교합니다.

외부에서 가져온 수업 함수는 아래 링크로 정의 행과 설명을 바로 확인합니다. 함수명 → docstring의 입력/출력 → 핵심 수식 → return 순서로 읽으세요. NumPy 공식 문서: [svd](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html), [eigh](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html).

| 함수 | 입력 | 출력·사용 이유 |
|---|---|---|
| teaching_matrix | 예제 이름 | 수정 가능한 작은 행렬 사본 |
| unit_vector | 0이 아닌 벡터 | 같은 방향, 길이1 |
| symmetric_eigen | 실수 대칭행렬 | 내림차순 values, 열별 vectors, 대입 잔차 |
| svd_steps | 행렬, 남길 rank | U/s/Vt/Gram/근사/오차/에너지 사전 |
| transform_stages | 2×2 행렬, 2×N 점 | 입력→Vt→Sigma→U의 네 배열 |
| direction_view | 예제, 각도 | 설명·표·화살표 그림 |
| stages_view | 예제 | 설명·단위원 변환 그림 |
| reconstruction_view | 예제, rank, 선택 report_fn | 설명·근사 표·비교 그림 |
| build_svd_app | 선택 report_fn | 웹 앱 Blocks; launch 별도 |

`svd_steps`의 energy_fraction은 입력 제곱합 보존 비율이며 추천 정확도가 아닙니다. 영행렬에서는 None입니다. `symmetric_eigen`을 일반 비대칭행렬에 쓰지 마세요.


In [ ]:
import os, sys, subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w06b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from luna_recsys.svd_foundations import teaching_matrix, unit_vector, symmetric_eigen, svd_steps, transform_stages
from luna_recsys.svd_lab import direction_view, stages_view, reconstruction_view, build_svd_app
from luna_recsys.model_comparison import definition_link
np.set_printoptions(precision=6,suppress=True)
print('Python',sys.version.split()[0], '고정 수업 버전',COURSE_VERSION)


In [ ]:
apis=[teaching_matrix,unit_vector,symmetric_eigen,svd_steps,transform_stages,
      direction_view,stages_view,reconstruction_view,build_svd_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 외적 한 항은 하나의 패턴

$A=\sum_i\sigma_i u_i v_i^\top$입니다. `np.outer(u,v)`는 열벡터×행벡터로 행렬을 만듭니다. 내적 `u@v`는 숫자 하나여서 다릅니다. 첫 항만 남긴 rank-1 결과를 먼저 예상하세요.


### B1 · 예상하기: [[5,1],[1,5]]의 첫 항은 어느 원소를 크게 만들까? 모든 원소가3이 되는 이유를 설명하세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
b1_reason=''


In [ ]:
A=teaching_matrix();d=svd_steps(A)
patterns=[d['s'][i]*np.outer(d['U'][:,i],d['Vt'][i,:]) for i in range(2)]
print('첫 패턴:\n',patterns[0]);print('둘째 패턴:\n',patterns[1]);print('합:\n',sum(patterns))
assert np.allclose(patterns[0],np.full((2,2),3))
assert np.allclose(sum(patterns),A)
rows=[]
for k in range(3):
    r=svd_steps(A,k)
    rows.append({'k':k,'Frobenius error':r['error'],'entry RMSE':r['error']/np.sqrt(A.size),'energy fraction':r['energy_fraction']})
    assert np.isclose(r['error']**2,np.sum(r['s'][k:]**2))
display(pd.DataFrame(rows))
txt,table,fig=reconstruction_view('main',1);print(txt);display(table);display(fig)


### B2 · 오차 계산: 차이 [[2,-2],[-2,2]]의 Frobenius 오차와 원소 RMSE를 구하세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
b2_fro=None
b2_rmse=None


## 2. 직사각형·rank 부족·영행렬

축소 SVD에서는 q=min(m,n)입니다. U는 m×q, s는 길이q, Vh는 q×n입니다. 특이값0에는 `Av/sigma`를 적용하지 않습니다. 직사각형 U에서 UᵀU와 UUᵀ를 혼동하지 마세요.


In [ ]:
for name in ['rectangular','deficient','zero']:
    r=svd_steps(teaching_matrix(name))
    print(name,'A/U/s/Vt:',r['A'].shape,r['U'].shape,r['s'].shape,r['Vt'].shape)
    print('sigma=',r['s'],'rank=',r['numerical_rank'],'energy=',r['energy_fraction'])
    print('sigma>0인 열만 Av/sigma:',r['positive_left'])
    assert np.allclose(r['U']@np.diag(r['s'])@r['Vt'],r['A'])
    assert np.isfinite(r['reconstruction']).all()
rect=svd_steps(teaching_matrix('rectangular'))
print('U.T@U:\n',rect['U'].T@rect['U'])
print('U@U.T:\n',rect['U']@rect['U'].T)


### B3 · 0으로 나누는 오류: s=[5,0]일 때 두 열 모두 Av/s로 계산하면 왜 안 되는가?

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
b3_reason=''


## 3. 미관측 0 대입의 반례

아래는 정답을 알고 일부러 한 칸을 감춘 작은 실험입니다. 실무의 실제 미관측 정답은 모릅니다. 0을 넣은 입력 M을 정확히 복원해도, 숨긴 정답1을 복원한 것은 아닙니다.


In [ ]:
truth=teaching_matrix();M=truth.copy();hidden=(0,1);M[hidden]=0
full=svd_steps(M,2);low=svd_steps(M,1)
print('원본 truth:\n',truth,'\n0 대입 입력 M:\n',M)
for name,r in [('rank-1',low),('rank-2',full)]:
    print(name,'입력 재구성 오차=',r['error'],'숨긴 칸 예측=',r['reconstruction'][hidden],
          '숨긴 칸 절대오차=',abs(r['reconstruction'][hidden]-truth[hidden]))
assert np.isclose(full['error'],0,atol=1e-10)
assert np.isclose(abs(full['reconstruction'][hidden]-truth[hidden]),1)


### B4 · 두 평가를 구별하기: 입력 재구성 오차0이면 추천 RMSE도0인가? 위 수치를 근거로 답하세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
b4_reason=''


## 4. 앱과 자신의 코드를 연결하기

아래 `my_report`는 앱의 ‘재구성 해석과 나의 문장’에 표시됩니다. **B5:** `prefix`를 자신의 문장으로 바꾸고, `r['rank']` 또는 `r['numerical_rank']`를 추가해 현재 k와 원본 rank를 구별해 보세요. 앱을 다시 실행해 자신의 문장이 바뀌었는지 확인합니다.

관찰 과제: main에서 k=0→1→2, negative에서 각도0°→90°, deficient와 zero를 각각 선택하세요. 예상을 먼저 쓰고 숫자·그림으로 설명을 수정합니다. 영행렬의 보존 비율을 100%로 표시하지 않는 이유도 생각하세요.


In [ ]:
def my_report(r):
    prefix='내 실험 기록: 입력 행렬을 얼마나 잘 재구성했는지 확인했습니다.'
    return prefix+f" Frobenius 오차={r['error']:.3f}."
print(my_report(svd_steps(A,1)))
text,_,figure=reconstruction_view('main',1,report_fn=my_report)
assert '내 실험 기록' in text
print(text);display(figure)
b5_observation=''


## 5. 출구 질문

1. rank-1과 첫 번째 원소만 남기는 것은 어떻게 다른가?
2. 보존 에너지69.23%를 추천 정확도69.23%라고 부를 수 있는가?
3. 직사각형3×2의 축소 U는 몇 행 몇 열인가?
4. 특이값0이 있다는 말은 어떤 입력 방향에 어떤 일이 일어난다는 뜻인가?
5. Surprise의 SVD와 `np.linalg.svd`를 구별하는 핵심은 무엇인가?

계산 문제편 Q6–Q9와 연결해 복습하세요. 아래 앱이 이 노트북의 마지막 활동입니다.


## 마지막 활동 · 웹 애플리케이션 실행

Colab에서 아래 셀을 실행한 뒤 나온 화면 또는 `gradio.live` 링크를 엽니다. 링크는 현재 실행 중인 런타임에 의존하며 영구 배포 주소가 아닙니다. 종료되면 이 셀을 다시 실행하세요. 공유 화면에는 수업용 합성 행렬만 사용합니다.

‘지난 MF 실험’에는 기존 학습·검증·설정 비교 기능이 있습니다. **5×5 합성18관측** 복습 모드이며 MovieLens 결과가 아닙니다. 새 SVD 세 탭과 함께 MF의 평가 목적을 비교하세요. 로컬 실행 검증은 callback·구성 검사를 하며, Colab 화면 검증과 구별됩니다.


In [ ]:
assert np.isclose(svd_steps(teaching_matrix(),1)['error'],4)
assert np.isclose(svd_steps(M,2)['reconstruction'][hidden],0,atol=1e-10)
assert svd_steps(teaching_matrix('zero'))['energy_fraction'] is None
assert np.isfinite(svd_steps(teaching_matrix('deficient'))['positive_left']).all()
assert '고유방향입니다' in direction_view('main',45)[0]
assert '오차' in reconstruction_view('main',1,report_fn=my_report)[0]
print('W06B-B RUNTIME CHECK PASSED: rank, zero, held-out error, UI callbacks.')


In [ ]:
app=build_svd_app(report_fn=my_report)
if IN_COLAB:
    app.launch(share=True,debug=False,prevent_thread_lock=True)
else:
    print('앱 구성 및 callback 준비 완료. Colab에서는 이 셀에 UI와 공유 링크가 열립니다.')
